In [4]:
import time

'''
Quality:
  CoP - Context Precision: fraction of retrieved chunks that are golden
  CiP - Citation Precision: fraction of cited sources that are correct
  RR  - Refusal Rate: fraction of correct refusal decisions (maximize)
  HR  - Hallucination Rate: fraction of answers with unsupported claims (minimize)
Latency:
  RT  - mean retrieval time (s)
'''

# golden data: query -> set of golden chunk IDs
golden_pairs = {
    # "query text": {"chunk_id_1", "chunk_id_2"},
}

def get_citation(chunk):
    """Extract a comparable ID from a chunk (depends on your chunk schema)."""
    return chunk["id"]  # or chunk.metadata["source"], etc.

def context_precision(query, retrieved_chunks, golden_pairs):
    if not retrieved_chunks:
        return 0.0
    correct = sum(1 for c in retrieved_chunks
                  if get_citation(c) in golden_pairs[query])
    return correct / len(retrieved_chunks)

def citation_precision(cited_ids, golden_ids):
    """Of the citations the model produced, how many are golden?"""
    if not cited_ids:
        return 0.0
    return sum(1 for c in cited_ids if c in golden_ids) / len(cited_ids)

def refusal_score(did_refuse, should_refuse):
    correct = sum(1 for d, s in zip(did_refuse, should_refuse) if d == s)
    return correct / len(should_refuse)

def hallucination_rate(judgments):
    """judgments: list of bools, True = answer contained hallucination.
    v0: fill by hand or with an LLM judge."""
    if not judgments:
        return 0.0
    return sum(judgments) / len(judgments)

def quality_score(s):
    return (0.25 * s['COP']
          + 0.25 * s['CIP']
          + 0.25 * s['RR']          # already "higher = better"
          + 0.25 * (1 - s['HR']))

def benchmark_v0(dataset, rag):
    latencies, cop_list, cip_list = [], [], []
    did_refuse, should_refuse, halluc = [], [], []

    for sample in dataset:
        q = sample['query']

        t0 = time.time()
        chunks = rag.retrieve(q)
        latencies.append(time.time() - t0)

        answer = rag.generate(q, chunks)  # your bug: used undefined `retrieved_chunks`

        cop_list.append(context_precision(q, chunks, golden_pairs))
        cip_list.append(citation_precision(answer.get('citations', []),
                                           golden_pairs[q]))
        did_refuse.append(answer.get('refused', False))
        should_refuse.append(sample['should_refuse'])
        halluc.append(judge_hallucination(q, answer, chunks))  # you supply this

    scores = {
        'COP': sum(cop_list) / len(cop_list),
        'CIP': sum(cip_list) / len(cip_list),
        'RR':  refusal_score(did_refuse, should_refuse),
        'HR':  hallucination_rate(halluc),
        'RT':  sum(latencies) / len(latencies),
    }
    return quality_score(scores), scores